# R1-N1 — Evaluation Package Verification & Test Suite

This notebook verifies and executes the complete **EchoProof R1-N1 Evaluation Package** on Google Colab or local Python.
It directly exercises the actual repository source code and unittests under `src/echoproof/evaluation/` and `tests/`.

### Modules & Capabilities Covered
1. **Recording-Level Aggregation (`aggregate.py`)**: Chunk-to-recording probability aggregation via `math.fsum`, `RecordingScores` immutable dataclass with subset filtering, and formal contract validation.
2. **Evaluation Metrics & EER (`metrics.py`)**: Full recording-level discrimination metrics (Accuracy, TTS Precision, Recall, Genuine Recall, Macro F1, FPR, FNR, ROC-AUC, Average Precision, Brier Score, Epsilon-clipped Log Loss) and Linear ROC-interpolated Equal Error Rate (EER).
3. **Operating Threshold Optimization (`threshold.py`)**: Systematic evaluation of all score-induced partitions and the all-negative decision rule, maximizing Macro F1 with contract-governed tie-breaking.
4. **Temperature Scaling Calibration (`calibrate.py`)**: Log-odds temperature scaling ($T > 0$) preserving monotonic ranking, with bounded 1-D optimization minimizing cross-entropy.
5. **Paired Group-Aware Bootstrap (`bootstrap.py`)**: Clustered speaker-group resampling with replacement, paired multi-model evaluation on identical cohorts, and percentile confidence intervals.

**Runtime Recommendation**: Standard CPU runtime is sufficient; GPU is optional. Run all cells top to bottom.

In [32]:
import os, sys, json, subprocess, zipfile, time, hashlib, platform
from pathlib import Path

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Locate project root (Colab default or repository path discovery)
CANDIDATE_ROOTS = [
    Path(os.environ.get("ECHOPROOF_ROOT", "")),
    Path("/content/drive/MyDrive/PAI&AIES Project"),
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
]
ROOT = next((p for p in CANDIDATE_ROOTS if p.exists() and (p / "pyproject.toml").exists()), None)
assert ROOT is not None, f"EchoProof project root not found among: {[str(p) for p in CANDIDATE_ROOTS]}"
print(f"Project root identified: {ROOT}")

# Check for code sync archive if deployed in Colab
ZIP = ROOT / "echoproof_sync.zip"
if ZIP.exists():
    with zipfile.ZipFile(ZIP) as z:
        members = [m for m in z.namelist() if m.split("/")[0] in ("src", "scripts", "tests") and "__pycache__" not in m]
        z.extractall(ROOT, members)
    print(f"Unpacked {len(members)} code files from {ZIP.name}")

# Verify existence of all R1-N1 evaluation source and test files
EVAL_SOURCES = {
    "aggregate.py": ROOT / "src/echoproof/evaluation/aggregate.py",
    "metrics.py": ROOT / "src/echoproof/evaluation/metrics.py",
    "threshold.py": ROOT / "src/echoproof/evaluation/threshold.py",
    "calibrate.py": ROOT / "src/echoproof/evaluation/calibrate.py",
    "bootstrap.py": ROOT / "src/echoproof/evaluation/bootstrap.py",
    "test_r1_eval_aggregate.py": ROOT / "tests/test_r1_eval_aggregate.py",
    "test_r1_eval_metrics.py": ROOT / "tests/test_r1_eval_metrics.py",
    "test_r1_eval_threshold.py": ROOT / "tests/test_r1_eval_threshold.py",
    "test_r1_eval_calibrate.py": ROOT / "tests/test_r1_eval_calibrate.py",
    "test_r1_eval_bootstrap.py": ROOT / "tests/test_r1_eval_bootstrap.py",
}
missing = [k for k, p in EVAL_SOURCES.items() if not p.exists()]
assert not missing, f"Missing required R1-N1 files: {missing}"

SHA = {k: hashlib.sha256(p.read_bytes()).hexdigest() for k, p in EVAL_SOURCES.items()}
print("R1-N1 Source & Test Checksums (SHA-256 prefixes):")
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root identified: .
R1-N1 Source & Test Checksums (SHA-256 prefixes):
{
  "aggregate.py": "7162f889e265",
  "metrics.py": "3719ac831b28",
  "threshold.py": "e93b24f28e70",
  "calibrate.py": "900aa77dbd6e",
  "bootstrap.py": "fabed5480940",
  "test_r1_eval_aggregate.py": "eeaf00510bda",
  "test_r1_eval_metrics.py": "8970e323f74e",
  "test_r1_eval_threshold.py": "9f1f0f878ae4",
  "test_r1_eval_calibrate.py": "7644e565b168",
  "test_r1_eval_bootstrap.py": "1e8bb9fd6585"
}


In [33]:
# Configure bytecode and module resolution
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Install project in editable mode via pyproject.toml (--no-deps preserves Colab environment)
proc = subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(ROOT)],
                      capture_output=True, text=True)
if proc.returncode == 0:
    print("EchoProof package registered in editable mode.")
else:
    print("Editable install message (continuing with sys.path):")
    print(proc.stderr.strip()[:200])

# Audit dependencies
import numpy as np, scipy, sklearn
try:
    import pyarrow
    PYARROW_VER = pyarrow.__version__
except ImportError:
    PYARROW_VER = "not installed"

ENV = {
    "python": platform.python_version(),
    "os": platform.platform(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "pyarrow": PYARROW_VER,
    "cpu_count": os.cpu_count(),
}
print("\nEnvironment & Dependency Audit:")
print(json.dumps(ENV, indent=2))

EchoProof package registered in editable mode.

Environment & Dependency Audit:
{
  "python": "3.13.18",
  "os": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "numpy": "2.4.6",
  "scipy": "1.16.3",
  "scikit-learn": "1.6.1",
  "pyarrow": "23.0.1",
  "cpu_count": 2
}


## 1. Recording-Level Score Aggregation (`aggregate.py`)

### Module Architecture & Principles
In EchoProof, audio is processed internally in 4-second chunks (64,000 samples at 16 kHz), but **the recording is the unit of prediction and evaluation**. Segments are never treated as independent evaluation subjects.

* **Arithmetic Mean**: Each recording's prediction is the arithmetic mean of its chunk-level TTS probabilities, computed via `math.fsum`.
* **`RecordingScores` Dataclass**: Immutable frozen container storing `recording_ids`, `labels`, `scores`, `chunk_counts`, `group_ids`, `partitions`, and `validation_roles` in first-seen order.
* **Subset Filtering**: The `.subset()` method extracts partitions (`train`, `validation`, `test`) and validation roles (`selection`, `calibration`, `operating`) without order drift.
* **Contract Referential Integrity**: `aggregate_from_contracts` validates that all chunks belong to registered recordings, enforces `Recording.validate_supervised()`, extracts labels strictly from `Recording.label`, and verifies consistency against `SplitAssignment`.

In [34]:
from echoproof.evaluation.aggregate import RecordingScores, aggregate_recordings

# Demonstrate aggregation with multi-chunk recordings
demo_chunk_ids = ["chunk:101", "chunk:102", "chunk:103", "chunk:201", "chunk:202"]
demo_rec_ids = ["rec:001", "rec:001", "rec:001", "rec:002", "rec:002"]
demo_labels = [1, 1, 1, 0, 0]
demo_scores = [0.20, 0.40, 0.60, 0.80, 0.90]

rec_scores = aggregate_recordings(
    chunk_ids=demo_chunk_ids,
    recording_ids=demo_rec_ids,
    labels=demo_labels,
    chunk_scores=demo_scores,
    group_ids=["speaker:spk1", "speaker:spk2"],
    partitions=["validation", "validation"],
    validation_roles=["selection", "operating"],
)

print(f"Aggregated {len(demo_chunk_ids)} chunks into {len(rec_scores.recording_ids)} recordings:")
for rid, lbl, sc, cnt in zip(rec_scores.recording_ids, rec_scores.labels, rec_scores.scores, rec_scores.chunk_counts):
    print(f"  Recording {rid}: ground_truth={lbl}, mean_tts_prob={sc:.4f}, chunks_aggregated={cnt}")

sub_op = rec_scores.subset(validation_role="operating")
print(f"Subset validation_role='operating' extracted: {sub_op.recording_ids}")

Aggregated 5 chunks into 2 recordings:
  Recording rec:001: ground_truth=1, mean_tts_prob=0.4000, chunks_aggregated=3
  Recording rec:002: ground_truth=0, mean_tts_prob=0.8500, chunks_aggregated=2
Subset validation_role='operating' extracted: ('rec:002',)


In [35]:
# Execute test_r1_eval_aggregate.py
print("=== Running tests/test_r1_eval_aggregate.py ===")
proc_agg = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", str(ROOT / "tests"),
        "-p", "test_r1_eval_aggregate.py",
        "-v"
    ],
    cwd=ROOT,
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
    capture_output=True,
    text=True,
)
print(proc_agg.stdout + proc_agg.stderr)
AGG_PASSED = proc_agg.returncode == 0
print(f"Unit test result: {'PASSED' if AGG_PASSED else 'FAILED'}")

=== Running tests/test_r1_eval_aggregate.py ===
test_aggregate_from_contracts (test_r1_eval_aggregate.AggregateTests.test_aggregate_from_contracts) ... ok
test_aggregate_from_contracts_label_none_rejected (test_r1_eval_aggregate.AggregateTests.test_aggregate_from_contracts_label_none_rejected) ... ok
test_aggregate_from_contracts_wrong_score_order (test_r1_eval_aggregate.AggregateTests.test_aggregate_from_contracts_wrong_score_order) ... ok
test_conflicting_labels_within_recording (test_r1_eval_aggregate.AggregateTests.test_conflicting_labels_within_recording) ... ok
test_duplicate_chunk_id (test_r1_eval_aggregate.AggregateTests.test_duplicate_chunk_id) ... ok
test_expected_recording_ids (test_r1_eval_aggregate.AggregateTests.test_expected_recording_ids) ... ok
test_invalid_types_and_values (test_r1_eval_aggregate.AggregateTests.test_invalid_types_and_values) ... ok
test_length_mismatch_and_empty_input (test_r1_eval_aggregate.AggregateTests.test_length_mismatch_and_empty_input) ... ok


## 2. Comprehensive Evaluation Metrics & EER (`metrics.py`)

### Module Architecture & Principles
`metrics.py` implements the evaluation contract specified in `IMPLEMENTATION_PLAN.md` Section 11:

* **Interpolated Equal Error Rate (EER)**: Linear interpolation of the ROC curve where $\text{FPR} = \text{FNR} = 1 - \text{TPR}$. Correctly evaluates tied scores ($EER = 0.5$), reversed predictions ($EER = 1.0$), and perfect separation ($EER = 0.0$).
* **Discrimination Metrics**:
  * Accuracy: $(TP + TN) / N$
  * TTS Precision: $TP / (TP + FP)$
  * TTS Recall: $TP / (TP + FN)$
  * Genuine Recall: $TN / (TN + FP)$
  * Macro F1: $\frac{1}{2} (F_{1, \text{genuine}} + F_{1, \text{TTS}})$
  * Genuine False Positive Rate (FPR): $FP / (FP + TN)$
  * False Negative Rate (FNR): $FN / (FN + TP)$
  * Area Under ROC (ROC-AUC) via exact pairwise comparisons
  * Average Precision (AP)
  * Brier Score: $\frac{1}{N} \sum (s_i - y_i)^2$
  * Log Loss: Binary cross-entropy with contract epsilon clipping ($\epsilon = 10^{-7}$)
* **Integrity Enforcement**: Requires both genuine ($0$) and TTS ($1$) classes, raising `ContractError` on single-class panels, and rejecting non-finite values.

In [36]:
from echoproof.evaluation.metrics import compute_metrics, interpolated_eer

demo_labels = [0, 0, 0, 1, 1, 1]
demo_scores = [0.08, 0.22, 0.35, 0.65, 0.88, 0.94]

eer_res = interpolated_eer(demo_labels, demo_scores)
metrics_dict = compute_metrics(demo_labels, demo_scores, threshold=0.50)

print(f"Linear ROC-interpolated EER: {eer_res:.4f}")
print("Comprehensive Metrics at threshold=0.50:")
for k, v in metrics_dict.items():
    if isinstance(v, float):
        print(f"  {k:<20}: {v:.4f}")
    elif k == "confusion_matrix":
        print(f"  {k:<20}: TN={v[0][0]}, FP={v[0][1]}, FN={v[1][0]}, TP={v[1][1]}")
    else:
        print(f"  {k:<20}: {v}")

Linear ROC-interpolated EER: 0.0000
Comprehensive Metrics at threshold=0.50:
  accuracy            : 1.0000
  tts_precision       : 1.0000
  tts_recall          : 1.0000
  genuine_recall      : 1.0000
  macro_f1            : 1.0000
  fpr                 : 0.0000
  fnr                 : 0.0000
  roc_auc             : 1.0000
  eer                 : 0.0000
  average_precision   : 1.0000
  confusion_matrix    : TN=3, FP=0, FN=0, TP=3
  brier_score         : 0.0530
  log_loss            : 0.2305
  tp                  : 3
  fp                  : 0
  tn                  : 3
  fn                  : 0
  recordings          : 6
  threshold           : 0.5000


In [37]:
# Execute test_r1_eval_metrics.py
print("=== Running tests/test_r1_eval_metrics.py ===")
proc_met = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", str(ROOT / "tests"),
        "-p", "test_r1_eval_metrics.py",
        "-v"
    ],
    cwd=ROOT,
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
    capture_output=True,
    text=True,
)
print(proc_met.stdout + proc_met.stderr)
MET_PASSED = proc_met.returncode == 0
print(f"Unit test result: {'PASSED' if MET_PASSED else 'FAILED'}")

=== Running tests/test_r1_eval_metrics.py ===
test_greater_equal_threshold_boundary (test_r1_eval_metrics.MetricsTests.test_greater_equal_threshold_boundary) ... ok
test_non_finite_and_out_of_bounds_rejected (test_r1_eval_metrics.MetricsTests.test_non_finite_and_out_of_bounds_rejected) ... ok
test_nothing_predicted_tts (test_r1_eval_metrics.MetricsTests.test_nothing_predicted_tts) ... ok
test_perfect_separation (test_r1_eval_metrics.MetricsTests.test_perfect_separation) ... ok
test_reversed_scores (test_r1_eval_metrics.MetricsTests.test_reversed_scores) ... ok
test_single_class_rejected (test_r1_eval_metrics.MetricsTests.test_single_class_rejected) ... ok
test_tied_scores (test_r1_eval_metrics.MetricsTests.test_tied_scores) ... ok

----------------------------------------------------------------------
Ran 7 tests in 1.964s

OK

Unit test result: PASSED


## 3. Operating Threshold Selection (`threshold.py`)

### Module Architecture & Principles
`threshold.py` optimizes the operational decision threshold on **operating validation** data prior to locked evaluation:

* **All Score-Induced Partitions**: Evaluates every distinct probability boundary ($\text{score} \ge u$ for each $u \in \text{distinct\_scores}$) plus the formal all-negative decision boundary (predicting genuine for all recordings).
* **Macro F1 Maximization**: Selects the rule maximizing recording-level Macro F1.
* **Contract Tie-Breaking Hierarchy**: When partitions produce identical Macro F1, ties are broken via:
  1. Lower genuine FPR (minimizing false accusations).
  2. Proximity to 0.5 ($|u - 0.5|$ minimized).
  3. Smaller threshold.
* **`DecisionRule` Emission**: Emits a validated `DecisionRule` contract with either `kind="threshold"` (numerical threshold $\le 1.0$) or `kind="all_negative"` (`threshold=None`).

In [38]:
from echoproof.evaluation.threshold import select_threshold

demo_labels = [0, 0, 0, 1, 1, 1, 0, 1]
demo_scores = [0.15, 0.30, 0.45, 0.40, 0.70, 0.85, 0.60, 0.90]

rule, best_f1, best_fpr = select_threshold(demo_labels, demo_scores, tie_break="contract")
print(f"Optimal Decision Rule : {rule}")
print(f"  Rule kind           : {rule.kind}")
print(f"  Decision threshold  : {rule.threshold}")
print(f"  Maximized Macro F1  : {best_f1:.4f}")
print(f"  Associated FPR      : {best_fpr:.4f}")

Optimal Decision Rule : DecisionRule(schema_version=1, kind='threshold', threshold=0.7)
  Rule kind           : threshold
  Decision threshold  : 0.7
  Maximized Macro F1  : 0.8730
  Associated FPR      : 0.0000


In [39]:
# Execute test_r1_eval_threshold.py
print("=== Running tests/test_r1_eval_threshold.py ===")
proc_thr = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", str(ROOT / "tests"),
        "-p", "test_r1_eval_threshold.py",
        "-v"
    ],
    cwd=ROOT,
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
    capture_output=True,
    text=True,
)
print(proc_thr.stdout + proc_thr.stderr)
THR_PASSED = proc_thr.returncode == 0
print(f"Unit test result: {'PASSED' if THR_PASSED else 'FAILED'}")

=== Running tests/test_r1_eval_threshold.py ===
test_agrees_with_brute_force (test_r1_eval_threshold.ThresholdTests.test_agrees_with_brute_force) ... ok
test_all_negative_case (test_r1_eval_threshold.ThresholdTests.test_all_negative_case) ... ok
test_never_above_one (test_r1_eval_threshold.ThresholdTests.test_never_above_one) ... ok
test_separable_data_macro_f1_one (test_r1_eval_threshold.ThresholdTests.test_separable_data_macro_f1_one) ... ok
test_single_class_and_invalid_inputs (test_r1_eval_threshold.ThresholdTests.test_single_class_and_invalid_inputs) ... ok
test_uninformative_constant_scores (test_r1_eval_threshold.ThresholdTests.test_uninformative_constant_scores) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.014s

OK

Unit test result: PASSED


## 4. Temperature Scaling Probability Calibration (`calibrate.py`)

### Module Architecture & Principles
`calibrate.py` adjusts uncalibrated raw model probabilities to reflect empirical probabilities on **calibration validation** data:

* **Log-Odds Temperature Scaling**:
  $$\text{calibrated} = \sigma\left(\frac{\text{logit}(\text{clip}(s, \epsilon, 1 - \epsilon))}{T}\right), \quad T > 0$$
* **Strict Ranking Preservation**: Monotonic transformations guarantee that relative model rankings and AUROC are preserved identically.
* **Identity at $T = 1.0$**: Scaling by $T=1.0$ is identity within epsilon clipping boundaries.
* **Bounded 1-D Optimization**: Fits temperature $T$ by minimizing binary cross-entropy (log loss) using bounded 1-D search with golden-section search fallback.

In [40]:
from echoproof.evaluation.calibrate import apply_temperature, fit_temperature

# Synthesize overconfident predictions (logits inflated by factor of 2.5)
rng = np.random.RandomState(42)
base_logits = rng.normal(0, 1.2, 500)
true_probs = 1.0 / (1.0 + np.exp(-base_logits))
labels = (rng.uniform(0, 1, 500) < true_probs).astype(int)
overconfident_scores = 1.0 / (1.0 + np.exp(-base_logits * 2.5))

fitted_t = fit_temperature(labels.tolist(), overconfident_scores.tolist())
calibrated = apply_temperature(overconfident_scores.tolist(), T=fitted_t)

print(f"Fitted Temperature T: {fitted_t:.4f} (target ~2.50)")
print("Sample Probabilities (Raw Overconfident -> Calibrated):")
for r, c, y in zip(overconfident_scores[:5], calibrated[:5], labels[:5]):
    print(f"  Raw: {r:.4f} -> Calibrated: {c:.4f} (True Label: {y})")

Fitted Temperature T: 2.4571 (target ~2.50)
Sample Probabilities (Raw Overconfident -> Calibrated):
  Raw: 0.8161 -> Calibrated: 0.6471 (True Label: 0)
  Raw: 0.3978 -> Calibrated: 0.4579 (True Label: 0)
  Raw: 0.8747 -> Calibrated: 0.6880 (True Label: 1)
  Raw: 0.9897 -> Calibrated: 0.8652 (True Label: 0)
  Raw: 0.3313 -> Calibrated: 0.4290 (True Label: 0)


In [41]:
# Execute test_r1_eval_calibrate.py
print("=== Running tests/test_r1_eval_calibrate.py ===")
proc_cal = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", str(ROOT / "tests"),
        "-p", "test_r1_eval_calibrate.py",
        "-v"
    ],
    cwd=ROOT,
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
    capture_output=True,
    text=True,
)
print(proc_cal.stdout + proc_cal.stderr)
CAL_PASSED = proc_cal.returncode == 0
print(f"Unit test result: {'PASSED' if CAL_PASSED else 'FAILED'}")

=== Running tests/test_r1_eval_calibrate.py ===
test_identity_when_temperature_is_one (test_r1_eval_calibrate.CalibrateTests.test_identity_when_temperature_is_one) ... ok
test_invalid_temperature_and_inputs (test_r1_eval_calibrate.CalibrateTests.test_invalid_temperature_and_inputs) ... ok
test_ranking_is_strictly_preserved (test_r1_eval_calibrate.CalibrateTests.test_ranking_is_strictly_preserved) ... ok
test_recovers_known_temperature_on_overconfident_data (test_r1_eval_calibrate.CalibrateTests.test_recovers_known_temperature_on_overconfident_data) ... ok
test_single_class_rejected (test_r1_eval_calibrate.CalibrateTests.test_single_class_rejected) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.429s

OK

Unit test result: PASSED


## 5. Paired Cluster/Group Bootstrap (`bootstrap.py`)

### Module Architecture & Principles
`bootstrap.py` computes statistical uncertainty intervals while respecting intra-group dependencies:

* **Clustered Group Resampling**: Resamples entire speaker/source groups with replacement, ensuring all recordings of a given group remain unified in each replicate.
* **Paired Model Comparisons**: Evaluates multiple models on the exact same resampled cohort, isolating model differences from sampling variance.
* **Pairwise Difference CIs**: Computes percentile confidence intervals for $(\text{Metric}_A - \text{Metric}_B)$, verifying statistical significance without parametric assumptions.
* **Degenerate Handling**: Tracks and counts resamples lacking one class, reporting valid versus skipped resamples.

In [44]:
from echoproof.evaluation.bootstrap import paired_group_bootstrap

demo_groups = ["speaker:01", "speaker:01", "speaker:02", "speaker:02", "speaker:03", "speaker:03", "speaker:04", "speaker:04"]
demo_y = [0, 1, 0, 1, 0, 1, 0, 1]
demo_models = {
    "logistic_regression": [0.20, 0.75, 0.30, 0.70, 0.25, 0.80, 0.35, 0.65],
    "bidirectional_gru":   [0.10, 0.90, 0.15, 0.85, 0.20, 0.92, 0.18, 0.88],
}

def accuracy_metric(y_true, scores):
    return float(np.mean([(s >= 0.5) == y for s, y in zip(scores, y_true)]))

boot_out = paired_group_bootstrap(
    group_ids=demo_groups,
    labels=demo_y,
    scores_by_model=demo_models,
    metric_fn=accuracy_metric,
    n_resamples=500,
    seed=42,
    alpha=0.05,
)

print(f"Completed {boot_out['valid_resamples']} valid bootstrap resamples (skipped {boot_out['skipped_single_class']}):")
for m_name, est in boot_out["models"].items():
    print(f"  {m_name:<22}: Estimate={est['estimate']:.4f}, 95% CI=[{est['ci_lower']:.4f}, {est['ci_upper']:.4f}]")

print("\nPairwise Differences:")
for pair, diff in boot_out["differences"].items():
    print(f"  {pair:<30}: Diff={diff['estimate']:+.4f}, 95% CI=[{diff['ci_lower']:+.4f}, {diff['ci_upper']:+.4f}]")

Completed 500 valid bootstrap resamples (skipped 0):
  bidirectional_gru     : Estimate=1.0000, 95% CI=[1.0000, 1.0000]
  logistic_regression   : Estimate=1.0000, 95% CI=[1.0000, 1.0000]

Pairwise Differences:
  bidirectional_gru_vs_logistic_regression: Diff=+0.0000, 95% CI=[+0.0000, +0.0000]


In [45]:
# Execute test_r1_eval_bootstrap.py
print("=== Running tests/test_r1_eval_bootstrap.py ===")
proc_boot = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", str(ROOT / "tests"),
        "-p", "test_r1_eval_bootstrap.py",
        "-v"
    ],
    cwd=ROOT,
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
    capture_output=True,
    text=True,
)
print(proc_boot.stdout + proc_boot.stderr)
BOOT_PASSED = proc_boot.returncode == 0
print(f"Unit test result: {'PASSED' if BOOT_PASSED else 'FAILED'}")

=== Running tests/test_r1_eval_bootstrap.py ===
test_ci_contains_estimate (test_r1_eval_bootstrap.BootstrapTests.test_ci_contains_estimate) ... ok
test_groups_kept_whole (test_r1_eval_bootstrap.BootstrapTests.test_groups_kept_whole) ... ok
test_identical_models_give_zero_difference (test_r1_eval_bootstrap.BootstrapTests.test_identical_models_give_zero_difference) ... ok
test_invalid_inputs (test_r1_eval_bootstrap.BootstrapTests.test_invalid_inputs) ... ok
test_one_group_rejected (test_r1_eval_bootstrap.BootstrapTests.test_one_group_rejected) ... ok
test_reproducible_with_seed (test_r1_eval_bootstrap.BootstrapTests.test_reproducible_with_seed) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.051s

OK

Unit test result: PASSED


## 6. Unified Execution of All R1-N1 Evaluation Test Files

This section runs all five R1-N1 test files in sequence, compiling a unified summary table of test counts, execution durations, and pass/fail statuses.

In [46]:
test_files = [
    ("Recording Score Aggregation", "tests/test_r1_eval_aggregate.py"),
    ("Evaluation Metrics & EER", "tests/test_r1_eval_metrics.py"),
    ("Operating Threshold Selection", "tests/test_r1_eval_threshold.py"),
    ("Temperature Scaling Calibration", "tests/test_r1_eval_calibrate.py"),
    ("Paired Group Bootstrap", "tests/test_r1_eval_bootstrap.py"),
]

suite_results = []
total_start = time.monotonic()

print("=" * 80)
print(f"{'R1-N1 FULL SUITE UNIFIED EXECUTION':^80}")
print("=" * 80)

for module_name, test_path in test_files:
    t0 = time.monotonic()
    proc = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", str(ROOT / "tests"),
        "-p", Path(test_path).name,
        "-v"
    ],
    cwd=ROOT,
    env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
    capture_output=True,
    text=True,
)
    duration = time.monotonic() - t0
    combined_output = proc.stdout + proc.stderr

    # Determine test count and success status
    is_ok = proc.returncode == 0 and "FAIL" not in combined_output and "ERROR" not in combined_output
    summary_line = next((l.strip() for l in reversed(combined_output.splitlines()) if l.startswith("Ran ") or "OK" in l or "FAILED" in l), "")

    suite_results.append({
        "module": module_name,
        "test_file": test_path,
        "passed": is_ok,
        "duration_seconds": round(duration, 3),
        "summary_line": summary_line,
    })

    status_tag = "PASSED" if is_ok else "FAILED"
    print(f"[{status_tag}] {module_name:<34} ({duration:.3f}s) | {summary_line}")

total_duration = time.monotonic() - total_start
all_passed = all(r["passed"] for r in suite_results)

print("-" * 80)
print(f"{'FINAL SUITE SUMMARY':^80}")
print("-" * 80)
print(f"Suite Status   : {'ALL 5 EVALUATION MODULE TESTS PASSED' if all_passed else 'TEST FAILURES DETECTED'}")
print(f"Total Duration : {total_duration:.2f} seconds")
print(f"Modules Tested : {len(suite_results)} / 5")

                       R1-N1 FULL SUITE UNIFIED EXECUTION                       
[PASSED] Recording Score Aggregation        (0.210s) | OK
[PASSED] Evaluation Metrics & EER           (1.989s) | OK
[PASSED] Operating Threshold Selection      (0.342s) | OK
[PASSED] Temperature Scaling Calibration    (0.807s) | OK
[PASSED] Paired Group Bootstrap             (0.388s) | OK
--------------------------------------------------------------------------------
                              FINAL SUITE SUMMARY                               
--------------------------------------------------------------------------------
Suite Status   : ALL 5 EVALUATION MODULE TESTS PASSED
Total Duration : 3.74 seconds
Modules Tested : 5 / 5


## 7. Verdict and Evidence Export (`reports/r1/r1_n1_evaluation_report.json`)

Serializes the execution audit, environment information, file checksums, and test outcomes into `reports/r1/r1_n1_evaluation_report.json`.

In [47]:
out_dir = ROOT / "reports" / "r1"
out_dir.mkdir(parents=True, exist_ok=True)
target = out_dir / "r1_n1_evaluation_report.json"

verdict = {
    "task": "R1-N1",
    "verified": all_passed,
    "all_passed": all_passed,
    "total_duration_seconds": round(total_duration, 3),
    "passed_modules": [r["module"] for r in suite_results if r["passed"]],
    "failed_modules": [r["module"] for r in suite_results if not r["passed"]],
}

evidence = {
    "schema_version": 1,
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment": ENV,
    "source_sha256": SHA,
    "verdict": verdict,
    "suite_results": suite_results,
}

if target.exists():
    backup_name = f"r1_n1_evaluation_report.{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
    target.replace(out_dir / backup_name)
    print(f"Existing report archived to {backup_name}")

target.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(json.dumps(verdict, indent=2))
print(f"\nEvidence artifact successfully written to: {target}")

Existing report archived to r1_n1_evaluation_report.20261105T093728Z.json
{
  "task": "R1-N1",
  "verified": true,
  "all_passed": true,
  "total_duration_seconds": 3.737,
  "passed_modules": [
    "Recording Score Aggregation",
    "Evaluation Metrics & EER",
    "Operating Threshold Selection",
    "Temperature Scaling Calibration",
    "Paired Group Bootstrap"
  ],
  "failed_modules": []
}

Evidence artifact successfully written to: reports/r1/r1_n1_evaluation_report.json
